# CyberSOC Arena - 5-minute Colab demo

*An OpenEnv environment where an LLM acts as a Tier-2 SOC analyst.*

**This notebook is a `see-and-verify` demo for judges**, not the headline training run. It does five things end-to-end on a free Colab CPU runtime in roughly 5 minutes:

1. **Install** the CyberSOC Arena package from GitHub.
2. **Connect** to the live HF Space and confirm the env is healthy.
3. **Walk one episode** of the marquee 20-step `long_horizon_apt` scenario, step-by-step, against the live Space's WebSocket session.
4. **View the trained model's BEFORE/AFTER plots** loaded directly from the model repo.
5. **Run a 30-second numpy REINFORCE micro-train** against the local env, just to demonstrate the env-driven training loop in the simplest possible way.

The headline GRPO training run was on a Hugging Face Jobs **L40S 48GB** for 2 hr (Qwen2.5-1.5B-Instruct + LoRA), pushed to <https://huggingface.co/amit51/cybersoc-arena-qwen2.5-1.5b-grpo>. To reproduce that run yourself, see [`scripts/run_hf_job_l40s.sh`](https://github.com/AmitChowdary122/cyber-openenv/blob/main/scripts/run_hf_job_l40s.sh) on GitHub.

Submission Space: <https://huggingface.co/spaces/amit51/cybersoc-arena>
Mini-blog: [BLOG.md on the Space](https://huggingface.co/spaces/amit51/cybersoc-arena/blob/main/BLOG.md)

---

## 1. Install the CyberSOC Arena package

We install directly from GitHub so this notebook always runs the same code that's on the HF Space.

In [ ]:
!pip install -q --upgrade pip
!pip install -q git+https://github.com/AmitChowdary122/cyber-openenv.git
!pip install -q 'openenv-core>=0.2.3' websockets matplotlib numpy

In [ ]:
import cybersoc_arena
from cybersoc_arena import (
    CyberSOCEnv, CurriculumEnv, CyberAction, CyberSOCRubric,
    SCENARIO_TYPES, TIERS, __version__,
)

print(f'cybersoc_arena v{__version__} installed.')
print(f'Scenarios available  : {SCENARIO_TYPES}')
print(f'Curriculum tiers     : {[t.name for t in TIERS]}')
print(f'Composable rubric    : CyberSOCRubric (RFC 004), 17 leaf components')

## 2. Connect to the live HF Space

The env is deployed at <https://amit51-cybersoc-arena.hf.space>. We hit `/health` to confirm it's live, then call `/reset` once via the synchronous client to verify the JSON surface.

In [ ]:
import requests

SPACE_URL = 'https://amit51-cybersoc-arena.hf.space'

health = requests.get(f'{SPACE_URL}/health', timeout=20).json()
print(f'/health -> {health}')

# Stateless one-shot reset (no episode state -- multi-step needs WebSocket)
reset = requests.post(
    f'{SPACE_URL}/reset',
    json={'seed': 314, 'scenario_type': 'long_horizon_apt'},
    timeout=30,
).json()
print(f'\nAlert     : {reset["observation"]["alert"]["summary"][:120]}...')
print(f'Severity  : {reset["observation"]["alert"]["severity"]}')
print(f'Hosts     : {reset["observation"]["asset_inventory"]["hosts"][:5]}')
print(f'Visible IP: {reset["observation"]["asset_inventory"]["visible_ips"][:3]}')
print(f'Step budget: {reset["observation"]["step_budget"]} steps')

## 3. Walk one episode of `long_horizon_apt` step-by-step (locally)

Multi-step episodes need persistent state, which the stateless HTTP `/reset`/`/step` doesn't preserve (each request creates a fresh env by OpenEnv design). The proper way is the WebSocket `EnvClient` -- but for a clean demo we'll just instantiate the env locally and walk it with a small SOC-analyst heuristic that picks the most informative tool at each step.

This is the same heuristic used by `train_reinforce.py`. It's not a trained model -- it's a hand-coded baseline -- but it lets you watch the env produce evidence and a final attribution.

In [ ]:
from cybersoc_arena import CyberSOCEnv, CyberAction

env = CyberSOCEnv()
obs = env.reset(seed=314, scenario_type='long_horizon_apt')
print(f'Reset -> {obs.alert.summary[:100]}...')
print(f'Visible IPs: {obs.asset_inventory.visible_ips[:3]}')
print(f'Step budget: {obs.step_budget}\n')
print('=' * 70)

# Simple SOC-analyst heuristic: query_logs on each visible IP, then correlate,
# then identify the most-implicated IP
evidence_count_so_far = 0
for step in range(20):
    if obs.done:
        break
    # Pick the next action based on what we've gathered
    if step < 4 and step < len(obs.asset_inventory.visible_ips):
        a = CyberAction(action_type='query_logs',
                        ip=obs.asset_inventory.visible_ips[step])
    elif step < 6 and obs.asset_inventory.hosts:
        a = CyberAction(action_type='inspect_endpoint',
                        host=obs.asset_inventory.hosts[step % len(obs.asset_inventory.hosts)])
    elif step == 6:
        a = CyberAction(action_type='check_threat_intel',
                        ip=obs.asset_inventory.visible_ips[0])
    elif step < 11 and obs.asset_inventory.visible_ips:
        a = CyberAction(action_type='correlate_events',
                        entity=obs.asset_inventory.visible_ips[step % len(obs.asset_inventory.visible_ips)])
    else:
        # Commit: identify the IP that appears most in collected evidence
        from collections import Counter
        ip_mentions = Counter()
        for ev in obs.evidence_collected:
            for ip in obs.asset_inventory.visible_ips:
                if ip in ev.finding:
                    ip_mentions[ip] += 1
        target = ip_mentions.most_common(1)[0][0] if ip_mentions else obs.asset_inventory.visible_ips[0]
        a = CyberAction(action_type='identify_attacker', ip=target)

    obs = env.step(a)
    new_ev = obs.evidence_count - evidence_count_so_far
    evidence_count_so_far = obs.evidence_count
    new_finding = obs.evidence_collected[-1].finding[:80] if new_ev else ''
    print(f'Step {step:2d} | {a.action_type:20s} -> reward {obs.reward:+.3f}'
          f' | evidence: {obs.evidence_count} (+{new_ev})'
          f'{"  | " + new_finding if new_finding else ""}')

print('=' * 70)
print(f'\nEpisode done. Final reward: {obs.reward:+.2f}')
print(f'Total evidence collected: {obs.evidence_count}')
print(f'Reward breakdown (last step): {obs.info.get("breakdown", {})}')

## 4. View the trained model's BEFORE/AFTER plots

The headline GRPO training run was on Qwen2.5-1.5B-Instruct + LoRA on a Hugging Face Jobs L40S 48GB for 2 hr (480 prompts x 3 epochs x 8 generations = 360 GRPO steps). The plots below are loaded directly from the trained-model repo at <https://huggingface.co/amit51/cybersoc-arena-qwen2.5-1.5b-grpo>.

In [ ]:
from IPython.display import Image, Markdown, display

MODEL_REPO_RAW = 'https://huggingface.co/amit51/cybersoc-arena-qwen2.5-1.5b-grpo/resolve/main'

display(Markdown('### GRPO training reward (the headline plot)'))
display(Image(url=f'{MODEL_REPO_RAW}/grpo_reward_curve.png'))
display(Markdown('*Mean per-step env reward across 720 GRPO log points. Climbs from ~-0.20 at init to a stable +0.15 to +0.40 band by step ~500 -- the policy is learning tool discipline directly from live env reward, with no SFT warm-start.*'))

display(Markdown('\n### Per-scenario BEFORE / AFTER training'))
display(Image(url=f'{MODEL_REPO_RAW}/grpo_baseline_compare.png'))
display(Markdown('*Held-out greedy rollout, 4 episodes per scenario, identical seeds. Lifts concentrate on the harder multi-evidence scenarios where cross-host correlation pays off: `multi_stage_chain` +0.40, `data_exfiltration` +0.31, `credential_stuffing` +0.13.*'))

display(Markdown('\n### GRPO surrogate loss (KL-regularized policy gradient)'))
display(Image(url=f'{MODEL_REPO_RAW}/grpo_loss_curve.png'))
display(Markdown('*For GRPO, this loss drifting up while reward goes up is **the correct signal**, not divergence: the loss measures how far the policy has moved from the frozen reference (the KL penalty). Magnitudes (1e-4 to 8e-4) are normal LoRA-GRPO scale.*'))

## 5. Watch the env teach a numpy policy in 30 seconds

GRPO on a 1.5B LLM needs an L40S; we don't have one in Colab. But the env's reward function is rich enough that even a tiny CPU-only REINFORCE meta-policy learns measurable behaviour in seconds. This is the simplest possible demonstration that the training loop **connects to the live env** and the agent **actually improves**.

We train a 4-action softmax meta-policy (INVESTIGATE / CORRELATE / IDENTIFY / CLOSE_BENIGN) over the 6-tier `CurriculumEnv` for 800 episodes (~25 sec on a free Colab CPU). The action targets are picked by the same SOC-analyst heuristic used above.

In [ ]:
import numpy as np
import time
from collections import Counter
from cybersoc_arena import CurriculumEnv, CyberAction

META_ACTIONS = ['INVESTIGATE', 'CORRELATE', 'IDENTIFY', 'CLOSE_BENIGN']
FEATURE_DIM = 6  # [evidence_count, remaining_steps_frac, num_visible_ips_norm, num_hosts_norm, has_correlated, attacker_ev_norm]

rng = np.random.default_rng(42)
W = rng.normal(0, 0.05, size=(FEATURE_DIM, len(META_ACTIONS)))   # policy weights
b = np.zeros(len(META_ACTIONS))
lr = 0.05

def featurise(obs):
    return np.array([
        min(obs.evidence_count / 5, 1.0),
        obs.remaining_steps / max(obs.step_budget, 1),
        min(len(obs.asset_inventory.visible_ips) / 5, 1.0),
        min(len(obs.asset_inventory.hosts) / 5, 1.0),
        float(any(a.action_type == 'correlate_events' for a in obs.action_history)),
        min(sum(1 for ev in obs.evidence_collected if 'attacker' in ev.finding.lower()
                or 'malware' in ev.finding.lower() or 'c2' in ev.finding.lower()) / 3, 1.0),
    ])

def softmax(x):
    e = np.exp(x - x.max()); return e / e.sum()

def pick_target_action(meta_idx, obs):
    """Map a meta-action + obs to a concrete CyberAction."""
    meta = META_ACTIONS[meta_idx]
    if meta == 'INVESTIGATE' and obs.asset_inventory.visible_ips:
        ip = obs.asset_inventory.visible_ips[obs.step % len(obs.asset_inventory.visible_ips)]
        tool = ['query_logs', 'investigate_ip', 'check_threat_intel'][obs.step % 3]
        return CyberAction(action_type=tool, ip=ip)
    if meta == 'CORRELATE' and obs.asset_inventory.visible_ips:
        return CyberAction(action_type='correlate_events',
                           entity=obs.asset_inventory.visible_ips[0])
    if meta == 'IDENTIFY':
        ip_mentions = Counter()
        for ev in obs.evidence_collected:
            for ip in obs.asset_inventory.visible_ips:
                if ip in ev.finding: ip_mentions[ip] += 1
        ip = ip_mentions.most_common(1)[0][0] if ip_mentions else (
            obs.asset_inventory.visible_ips[0] if obs.asset_inventory.visible_ips else '0.0.0.0')
        return CyberAction(action_type='identify_attacker', ip=ip)
    if meta == 'CLOSE_BENIGN':
        return CyberAction(action_type='close_as_benign', summary='no actionable evidence')
    # Fallback if state is empty
    return CyberAction(action_type='query_logs', ip='0.0.0.0')

cenv = CurriculumEnv(window=20)
rewards_history = []
tier_history = []

t0 = time.time()
for ep in range(800):
    obs = cenv.reset()
    log_probs, episode_features, episode_actions = [], [], []
    while not obs.done:
        f = featurise(obs)
        logits = f @ W + b
        probs = softmax(logits)
        a_idx = rng.choice(len(META_ACTIONS), p=probs)
        episode_features.append(f)
        episode_actions.append(a_idx)
        obs = cenv.step(pick_target_action(a_idx, obs))
    G = obs.reward     # final cumulative reward
    rewards_history.append(G)
    tier_history.append(cenv.tier)
    cenv.record_episode_reward(G)
    # REINFORCE update: gradient = sum_t (G * (one_hot(a_t) - probs_t))
    for f, a_idx in zip(episode_features, episode_actions):
        logits = f @ W + b
        probs = softmax(logits)
        oh = np.zeros_like(probs); oh[a_idx] = 1.0
        grad_logits = G * (oh - probs)
        W += lr * np.outer(f, grad_logits)
        b += lr * grad_logits

elapsed = time.time() - t0
print(f'Trained 800 episodes in {elapsed:.1f} sec ({elapsed*1000/800:.1f} ms/episode)')
print(f'Reward (first 100 ep mean) : {np.mean(rewards_history[:100]):+.3f}')
print(f'Reward (last  100 ep mean) : {np.mean(rewards_history[-100:]):+.3f}')
print(f'Curriculum tier reached    : {cenv.tier} ({TIERS[cenv.tier].name})')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Smoothed reward curve
window = 30
smoothed = np.convolve(rewards_history, np.ones(window)/window, mode='valid')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True,
                                gridspec_kw={'height_ratios': [1, 0.6]})
ax1.plot(rewards_history, color='#bbb', lw=0.6, label='per-episode reward')
ax1.plot(np.arange(window-1, len(rewards_history)), smoothed,
         color='#1b9e77', lw=1.8, label=f'rolling mean (window={window})')
ax1.axhline(0, color='#999', ls='--', lw=0.8)
ax1.set_ylabel('Episode reward')
ax1.set_title('CyberSOC Arena - 30-second numpy REINFORCE on CurriculumEnv (Colab CPU)')
ax1.legend(loc='upper left'); ax1.grid(alpha=0.3)

ax2.step(range(len(tier_history)), tier_history, where='post', color='#525252', lw=1.6)
ax2.set_xlabel('Episode #'); ax2.set_ylabel('Curriculum tier')
ax2.set_yticks(list(range(6)))
ax2.grid(alpha=0.3, axis='x')

fig.tight_layout()
plt.show()

print('\nEnv-driven training works on the simplest possible setup. The headline')
print('GRPO run on Qwen2.5-1.5B + L40S 48GB does the same thing with 360 GRPO')
print('steps and produces the BEFORE/AFTER plots in Cell 4.')

## What's next

- **Reproduce the L40S GRPO run on the $30 hackathon credit:** clone the repo, `bash scripts/run_hf_job_l40s.sh` -- ~2 hr, ~$3.60.
- **Drive your own LLM against the live env:** see [`scripts/train_hf_job.py`](https://github.com/AmitChowdary122/cyber-openenv/blob/main/scripts/train_hf_job.py) for the prompt rendering + tolerant action parser.
- **Read the writeup:** [BLOG.md on the Space](https://huggingface.co/spaces/amit51/cybersoc-arena/blob/main/BLOG.md).
- **Inspect the rubric tree:** `from cybersoc_arena import CyberSOCRubric; r = CyberSOCRubric(); print(list(r.named_rubrics()))` -- 17 leaf components, RFC 004 compliant.

---

*Built solo for the OpenEnv Hackathon Round 2 (Meta x Hugging Face x PyTorch, Bangalore 2026). Apache-2.0 licensed.*